### Salarios por entidad
En este notebook se realiza la limpieza de  los datos obtenidos del Conjuto de datos: Poblacion ocupada de la Encuesta Nacional de Ocupacion y Empleo (ENOE), realizada por el INEGI. Los datos descargados fueron obtenidos realizando la siguiente consulta:  "Población ocupada por Periodo de encuesta y sexo, según Entidad Federativa y Nivel de ingresos. E incluye datos de cada cuarto de anio desde 2005 y hasta 2025. 
Estos datos se pueden encontrar en la siguiente liga: https://www.inegi.org.mx/sistemas/olap/consulta/general_ver4/MDXQueryDatos_colores.asp?#Regreso&c= 

In [47]:
# importamos bibliotecas necesarias
import pandas as pd

In [48]:
file_path = '../data/INEGI_exporta_13_4_2026_19_18_18.csv'

with open(file_path, 'r', encoding='utf-8-sig', errors='replace') as f:
    lines = f.readlines()

In [49]:
entidades = [e.strip() for e in lines[7].strip().split(',')]
ingresos = [i.strip() for i in lines[9].strip().split(',')]

data = []

In [50]:
# Iterar sobre las filas de datos (empiezan en la línea 11)
for line in lines[11:]:
    if 'trimestre' not in line:
        continue
    
    cols = [c.strip() for c in line.strip().split(',')]
    periodo = cols[0]
    sexo = cols[1]
    
    # Ignorar la fila de "Total" porque queremos ver la comparativa Hombre vs Mujer
    if sexo == 'Total':
        continue 
        
    # Cruzar los valores iterando sobre las columnas
    for i in range(2, len(cols)):
        if i < len(entidades) and i < len(ingresos):
            entidad = entidades[i]
            ingreso = ingresos[i]
            val_raw = cols[i]
            
            # El INEGI manda: Valor|CV|Error. Separamos por el "|" y nos quedamos con el Valor
            val_est = val_raw.split('|')[0] if '|' in val_raw else val_raw
            
            try:
                val_est = float(val_est) if val_est else 0.0
            except ValueError:
                continue # Ignora celdas vacías o notas al pie
                
            # Descartar las columnas de totales generales para no duplicar datos
            if entidad != 'Total' and ingreso != 'Total':
                data.append({
                    'Periodo': periodo,
                    'Entidad': entidad,
                    'Sexo': sexo,
                    'Rango_Ingreso': ingreso,
                    'Poblacion': val_est
                })

In [51]:
df_clean = pd.DataFrame(data)

display(df_clean.head())

,Periodo,Entidad,Sexo,Rango_Ingreso,Poblacion
0,Cuarto trimestre del 2025,Aguascalientes,Hombre,Hasta un salario m�nimo,79307.0
1,Cuarto trimestre del 2025,Aguascalientes,Hombre,M�s de 1 hasta 2 salarios m�nimos,149512.0
2,Cuarto trimestre del 2025,Aguascalientes,Hombre,M�s de 2 hasta 3 salarios m�nimos,32839.0
3,Cuarto trimestre del 2025,Aguascalientes,Hombre,M�s de 3 hasta 5 salarios m�nimos,14624.0
4,Cuarto trimestre del 2025,Aguascalientes,Hombre,M�s de 5 salarios m�nimos,3913.0


Si bien esta tabla es lo suficientemente interpretable para cualquier persona, podemos hacer modificaciones de limpieza adicionales con el fin de facilitar la realizacion de graficas, e incluso aplicacion de modelos de ML.

In [52]:
df_clean['Anio'] = df_clean['Periodo'].str.extract(r'(\d{4})').astype(int)

# Extraer el Trimestre en formato I, II, III, IV
trimestre_map_roman = {
    'Primer': 'I',
    'Segundo': 'II',
    'Tercer': 'III',
    'Cuarto': 'IV'
}

# Diccionario auxiliar para el mes de inicio del trimestre
trimestre_map_num = {
    'Primer': '01', # Enero
    'Segundo': '04', # Abril
    'Tercer': '07', # Julio
    'Cuarto': '10'  # Octubre
}

In [53]:
# Extraemos la primera palabra (Ej. "Cuarto")
df_clean['Trimestre_texto'] = df_clean['Periodo'].str.split(' ').str[0]

In [54]:
# Mapeamos a números romanos
df_clean['Trimestre'] = df_clean['Trimestre_texto'].map(trimestre_map_roman)

In [56]:
# Crear la columna Fecha (mapeando a YYYY-MM-01) para gráficas de series de tiempo
meses = df_clean['Trimestre_texto'].map(trimestre_map_num)
df_clean['Fecha'] = pd.to_datetime(df_clean['Anio'].astype(str) + '-' + meses + '-01')

In [57]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37184 entries, 0 to 37183
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Periodo          37184 non-null  object        
 1   Entidad          37184 non-null  object        
 2   Sexo             37184 non-null  object        
 3   Rango_Ingreso    37184 non-null  object        
 4   Poblacion        37184 non-null  float64       
 5   Anio             37184 non-null  int32         
 6   Trimestre_texto  37184 non-null  object        
 7   Trimestre        37184 non-null  object        
 8   Fecha            37184 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int32(1), object(6)
memory usage: 2.4+ MB


In [58]:
print(df_clean.Rango_Ingreso.values)

['Hasta un salario m�nimo' 'M�s de 1 hasta 2 salarios m�nimos'
 'M�s de 2 hasta 3 salarios m�nimos' ... 'M�s de 5 salarios m�nimos'
 'No recibe ingresos' 'No especificado']


In [59]:
# Mapeo de variables categoricas (se puede skipear)
ingreso_map = {
    'No recibe ingresos': 0,
    'Hasta un salario m�nimo': 1,
    'M�s de 1 hasta 2 salarios m�nimos': 2,
    'M�s de 2 hasta 3 salarios m�nimos': 3,
    'M�s de 3 hasta 5 salarios m�nimos': 4,
    'M�s de 5 salarios m�nimos': 5,
    'No especificado': -1

}

In [61]:
df_clean['Rango_Ingreso_Num'] = df_clean['Rango_Ingreso'].map(ingreso_map)

display(df_clean)
# Limpieza final: tiramos la columna temporal que usamos para el mapeo
df_clean = df_clean.drop(columns=['Trimestre_texto'])

# Guardamos el CSV final listo para el Dashboard
df_clean.to_csv('../data/enoe_salarios_sexo_entidad.csv', index=False)

,Periodo,Entidad,Sexo,Rango_Ingreso,Poblacion,Anio,Trimestre_texto,Trimestre,Fecha,Rango_Ingreso_Num
0,Cuarto trimestre del 2025,Aguascalientes,Hombre,Hasta un salario m�nimo,79307.0,2025,Cuarto,IV,2025-10-01,1
1,Cuarto trimestre del 2025,Aguascalientes,Hombre,M�s de 1 hasta 2 salarios m�nimos,149512.0,2025,Cuarto,IV,2025-10-01,2
2,Cuarto trimestre del 2025,Aguascalientes,Hombre,M�s de 2 hasta 3 salarios m�nimos,32839.0,2025,Cuarto,IV,2025-10-01,3
3,Cuarto trimestre del 2025,Aguascalientes,Hombre,M�s de 3 hasta 5 salarios m�nimos,14624.0,2025,Cuarto,IV,2025-10-01,4
4,Cuarto trimestre del 2025,Aguascalientes,Hombre,M�s de 5 salarios m�nimos,3913.0,2025,Cuarto,IV,2025-10-01,5
...,...,...,...,...,...,...,...,...,...,...
37179,Primer trimestre del 2005,Zacatecas,Mujer,M�s de 2 hasta 3 salarios m�nimos,11209.0,2005,Primer,I,2005-01-01,3
37180,Primer trimestre del 2005,Zacatecas,Mujer,M�s de 3 hasta 5 salarios m�nimos,16115.0,2005,Primer,I,2005-01-01,4
37181,Primer trimestre del 2005,Zacatecas,Mujer,M�s de 5 salarios m�nimos,10704.0,2005,Primer,I,2005-01-01,5
37182,Primer trimestre del 2005,Zacatecas,Mujer,No recibe ingresos,25470.0,2005,Primer,I,2005-01-01,0
